In [ ]:
import pandas as pd

delete = pd.read_parquet("C:\qtri\Self-learning\SEACrowd\gated-continual-cartridges\experiments_longhealth\longhealth_patient1_10_og.parquet")

In [ ]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="qtris123/qwen2507_longhealth-p1-10_8192_512_no-cartridge_10-epochs",
    filename="cache_last.pt",
    local_dir="cartridges_pt/qwen3_512_p1"
)
print(f"File downloaded to: {file_path}")

file_path = hf_hub_download(
    repo_id="qtris123/qwen2507_longhealth-p1-10_8192_1024_no-cartridge_10-epochs",
    filename="cache_last.pt",
    local_dir="cartridges_pt/qwen3_1024_p1"
)
print(f"File downloaded to: {file_path}")


file_path = hf_hub_download(
    repo_id="qtris123/qwen2507_longhealth-p1-10_8192_2048_no-cartridge_10-epochs",
    filename="cache_last.pt",
    local_dir="cartridges_pt/qwen3_2048_p1"
)
print(f"File downloaded to: {file_path}")

file_path = hf_hub_download(
    repo_id="qtris123/llama_longhealth-p1-10_8192_512_no-cartridge_10-epochs",
    filename="cache_last.pt",
    local_dir="cartridges_pt/llama_512_p1"
)
print(f"File downloaded to: {file_path}")

file_path = hf_hub_download(
    repo_id="qtris123/llama_longhealth-p1-10_8192_1024_no-cartridge_10-epochs",
    filename="cache_last.pt",
    local_dir="cartridges_pt/llama_1024_p1"
)
print(f"File downloaded to: {file_path}")


file_path = hf_hub_download(
    repo_id="qtris123/llama_longhealth-p1-10_8192_2048_no-cartridge_10-epochs",
    filename="cache_last.pt",
    local_dir="cartridges_pt/llama_2048_p1"
)
print(f"File downloaded to: {file_path}")

File downloaded to: cartridges_pt/qwen3_1024_p1.pt/cache_last.pt


-------------------------------------

In [6]:
import csv
import sys

# Set the limit to the maximum possible for your system
csv.field_size_limit(sys.maxsize)

131072

In [7]:
import csv
import re
from pathlib import Path
from cartridges.structs import Conversation, write_conversations

def convert_longhealth_csv_to_parquet(csv_path: str, parquet_path: str):
    conversations = []
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            question_text = row["question"]
            correct = row["correct"]
            metadata = {
                "question_id": row["question_id"],
                "question_type": "longhealth_mc",
                "doc_source": "longhealth",
                "options": [
                    row["answer_a"], row["answer_b"], row["answer_c"],
                    row["answer_d"], row["answer_e"],
                ],
                "answer_location": row.get("answer_location", ""),
            }
            convo = Conversation(
                system_prompt="",
                messages=[
                    Conversation.Message(role="user", content=question_text,
                                         token_ids=None, top_logprobs=None),
                    Conversation.Message(role="assistant", content=correct,
                                         token_ids=None, top_logprobs=None),
                ],
                metadata=metadata,
                type="continual_eval",
            )
            conversations.append(convo)
    write_conversations(conversations, parquet_path)
    print(f"Wrote {len(conversations)} conversations to {parquet_path}")

In [8]:
convert_longhealth_csv_to_parquet("/home/vo43/cartridges/examples/longhealth/data/patient1_10.csv", "/home/vo43/cartridges/experiments_longhealth/longhealth_patient1_10_og.parquet")

Wrote 200 conversations to /home/vo43/cartridges/experiments_longhealth/longhealth_patient1_10_og.parquet


In [9]:
with open("/home/vo43/cartridges/experiments_longhealth/longhealth_patient1_10_og.parquet", "rb") as f:
    df = pd.read_parquet(f)


In [13]:
df.iloc[0]["messages"]

array([{'content': "Please answer the question below about the following patient: ID patient_07, Name: Linda Mayer, Birthday: 1948-12-01 00:00:00, Diagnosis: Breast carcinoma\n\n<question>\nConsidering Mrs. Mayer's medications that were present in 2019 and 2021, which drug's dosing frequency was reduced in the second regimen, likely indicating a clinical improvement?\n</question>\n\n<options>\nAspirin\nSimvastatin\nPantoprazole\nPrednisolone\nAcyclovir\n</options>\nYou should first think step by step. Then give your final answer exactly as it appears in the options. Your output should be in the following format: \n<thinking> {{YOUR_THOUGHT_PROCESS}} </thinking> \n\n<answer>\n{YOUR_ANSWER}\n</answer>", 'role': 'user', 'token_ids': None, 'top_logprobs': None},
       {'content': 'Prednisolone', 'role': 'assistant', 'token_ids': None, 'top_logprobs': None}],
      dtype=object)